# EOEPCA Notification and Automation Usage Notebook

The Notification and Automation Building Block is an event-driven workflow layer built on Knative. It turns things happening on the platform - a GitHub webhook, a Kubernetes API event - into [CloudEvents](https://cloudevents.io/), routes them through a broker, and lets you subscribe your own serverless functions to react to them.

This notebook walks through the deployed BB: checking it's up, watching events flow through the CloudEvents Player, turning a simulated GitHub webhook into a CloudEvent, and (optionally, if you have `kubectl` access to the cluster) wiring your own function to react to one.

## Setup

In [ ]:
import hashlib
import hmac
import json
import os
import sys
import uuid
from time import sleep, time

import requests

sys.path.append('../')
from modules.helpers import load_eoepca_state, test_cell, test_results

Load the `eoepca state` environment.

In [ ]:
load_eoepca_state()

In [ ]:
platform_domain = os.environ["INGRESS_HOST"]
http_scheme = os.environ["HTTP_SCHEME"]
github_webhook_secret = os.environ["NA_GITHUB_WEBHOOK_SECRET"]

webhook_source_url = f"{http_scheme}://webhooks.notifications.{platform_domain}"
cloudevents_player_url = f"{http_scheme}://cloudevents-player.notifications.{platform_domain}"

log_output_file = "notification-automation_log.json"

## Check the Building Block is up

The webhook source and CloudEvents player each get their own ingress. A quick health check on both confirms the chart is deployed and reachable before doing anything else.

In [ ]:
# endpoints_healthy
player_response = requests.get(cloudevents_player_url, timeout=10)
webhook_health_response = requests.get(f"{webhook_source_url}/health", timeout=10)

print(f"CloudEvents Player: {player_response.status_code}")
print(f"Webhook source health: {webhook_health_response.status_code} {webhook_health_response.text}")

assert player_response.status_code == 200
assert webhook_health_response.status_code == 200

## Watching events flow: the CloudEvents Player

The chart deploys a `default` Broker with a Trigger already subscribing the CloudEvents Player to it, and an API Server Source that turns Kubernetes `Event` objects in the `notifications` namespace into CloudEvents on that broker. That means the player is already showing live traffic with zero setup - useful both as a demo and as a general-purpose "what just happened in this namespace" view. Open `cloudevents_player_url` in a browser to watch it live, or pull the same data over its REST API as below.

In [ ]:
def fetch_events(size=50):
    response = requests.get(
        f"{cloudevents_player_url}/messages",
        params={"page": 0, "size": size},
        timeout=10,
    )
    response.raise_for_status()
    return response.json()

In [ ]:
# api_server_source_events
events = fetch_events()
print(f"CloudEvents Player has {len(events)} events recorded")
for event in events[:3]:
    print(f"  {event['eventType']} from {event['source']}")

assert len(events) > 0
assert any(event["eventType"].startswith("dev.knative.apiserver.") for event in events)

## Turning a GitHub webhook into a CloudEvent

The webhook source's `/github` endpoint is what you point a real GitHub repository's webhook settings at, using `NA_GITHUB_WEBHOOK_SECRET` (from `~/.eoepca/state`) as the webhook secret. GitHub signs every delivery with `X-Hub-Signature-256`, an HMAC-SHA256 of the raw body using that secret - the webhook source verifies it before accepting anything. We simulate a real delivery here rather than requiring a public repository.

In [ ]:
def signed_github_request(payload: dict, secret: str):
    body = json.dumps(payload).encode()
    signature = "sha256=" + hmac.new(secret.encode(), body, hashlib.sha256).hexdigest()
    headers = {
        "Content-Type": "application/json",
        "X-GitHub-Event": "ping",
        "X-GitHub-Delivery": str(uuid.uuid4()),
        "X-Hub-Signature-256": signature,
    }
    return body, headers

A request signed with the wrong secret is rejected outright - it never reaches the broker.

In [ ]:
# webhook_signature_rejected
body, headers = signed_github_request({"zen": "wrong secret"}, "not-the-real-secret")
response = requests.post(f"{webhook_source_url}/github", data=body, headers=headers, timeout=10)

print(f"Status: {response.status_code} {response.text}")
assert response.status_code == 401

A correctly-signed delivery is accepted immediately (`202`) and, moments later, shows up in the CloudEvents Player as an `org.eoepca.webhook.github.ping` event.

In [ ]:
# webhook_event_delivered
marker = str(uuid.uuid4())
payload = {"zen": "Automation over integration.", "hook_id": 1, "marker": marker}
body, headers = signed_github_request(payload, github_webhook_secret)

response = requests.post(f"{webhook_source_url}/github", data=body, headers=headers, timeout=10)
print(f"Delivery accepted: {response.status_code} {response.json()}")
assert response.status_code == 202

deadline = time() + 30
delivered = None
while time() < deadline and delivered is None:
    for event in fetch_events():
        if event.get("data", {}).get("marker") == marker:
            delivered = event
            break
    if delivered is None:
        sleep(2)

print(f"Delivered event: {json.dumps(delivered, indent=2) if delivered else 'not seen within 30s'}")
assert delivered is not None
assert delivered["eventType"] == "org.eoepca.webhook.github.ping"

## Creating an isolated broker

`default` (used above) already carries webhook and API-server events. Create your own broker when you want a separate event space - e.g. so your own triggers don't match unrelated platform events. This section needs `kubectl` access and is skipped otherwise.

In [ ]:
import shutil
import subprocess

kubectl_available = shutil.which("kubectl") is not None

isolated_broker_yaml = """\
apiVersion: eventing.knative.dev/v1
kind: Broker
metadata:
  name: notebook-demo
  namespace: notifications
"""

if kubectl_available:
    try:
        subprocess.run(["kubectl", "apply", "-f", "-"], input=isolated_broker_yaml, text=True, check=True)
        subprocess.run(
            ["kubectl", "wait", "--for=condition=Ready", "broker/notebook-demo", "-n", "notifications", "--timeout=60s"],
            check=True,
        )
        print("Broker 'notebook-demo' is Ready - independent of the 'default' broker used above.")
    finally:
        subprocess.run(["kubectl", "delete", "broker", "notebook-demo", "-n", "notifications", "--ignore-not-found"])
else:
    print("kubectl not available on PATH - skipping.")

## Reacting to events with your own function

This is the other half of the BB: subscribe a Knative Service to the broker and it fires whenever a matching event arrives. This section needs `kubectl` configured against the cluster - it's skipped automatically if `kubectl` isn't on the `PATH` (it's not part of the automated test above).

Deploy the standard Knative sample as a quick, disposable function, and subscribe it to the `default` broker filtered to a specific event type - a Trigger with no filter would also match the function's own plain-text response and fire itself again in a loop:

In [ ]:
import shutil
import subprocess

kubectl_available = shutil.which("kubectl") is not None
print(f"kubectl available: {kubectl_available}")

In [ ]:
hello_function_yaml = """\
apiVersion: serving.knative.dev/v1
kind: Service
metadata:
  name: hello-function
  namespace: notifications
spec:
  template:
    spec:
      containers:
        - image: gcr.io/knative-samples/helloworld-go
          env:
            - name: TARGET
              value: "EOEPCA Platform"
"""

hello_function_trigger_yaml = """\
apiVersion: eventing.knative.dev/v1
kind: Trigger
metadata:
  name: hello-function-trigger
  namespace: notifications
spec:
  broker: default
  filter:
    attributes:
      type: org.eoepca.demo.hello
  subscriber:
    ref:
      apiVersion: serving.knative.dev/v1
      kind: Service
      name: hello-function
"""

def kubectl_cleanup():
    subprocess.run(["kubectl", "delete", "trigger", "hello-function-trigger", "-n", "notifications", "--ignore-not-found"])
    subprocess.run(["kubectl", "delete", "ksvc", "hello-function", "-n", "notifications", "--ignore-not-found"])

if kubectl_available:
    try:
        subprocess.run(["kubectl", "apply", "-f", "-"], input=hello_function_yaml, text=True, check=True)
        subprocess.run(["kubectl", "wait", "--for=condition=Ready", "ksvc/hello-function", "-n", "notifications", "--timeout=180s"], check=True)
        subprocess.run(["kubectl", "apply", "-f", "-"], input=hello_function_trigger_yaml, text=True, check=True)
    except Exception:
        kubectl_cleanup()
        raise

The broker's own address is cluster-internal, so post the demo event from a pod inside the cluster and check the function's own logs for it:

In [ ]:
if kubectl_available:
    try:
        subprocess.run([
            "kubectl", "run", "curl-test", "--restart=Never", "--namespace=notifications",
            "--image=curlimages/curl:latest", "--command", "--",
            "curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
            "http://broker-ingress.knative-eventing.svc.cluster.local/notifications/default",
            "-H", "Ce-Id: notebook-demo-1",
            "-H", "Ce-Specversion: 1.0",
            "-H", "Ce-Type: org.eoepca.demo.hello",
            "-H", "Ce-Source: notebook-demo",
            "-H", "Content-Type: application/json",
            "-d", json.dumps({"message": "hello from the notebook"}),
        ], check=True)
        subprocess.run(["kubectl", "wait", "--for=condition=Ready", "pod/curl-test", "-n", "notifications", "--timeout=30s"], check=True)
        sleep(3)
        logs = subprocess.run(
            ["kubectl", "logs", "-n", "notifications", "-l", "serving.knative.dev/service=hello-function", "-c", "user-container", "--tail=5"],
            capture_output=True, text=True,
        )
        print(logs.stdout)
        assert "received a request" in logs.stdout
    finally:
        subprocess.run(["kubectl", "delete", "pod", "curl-test", "-n", "notifications", "--ignore-not-found"])
        kubectl_cleanup()

You should see a `helloworld: received a request` line.

## Results

In [ ]:
if test_results:
    for test, result in test_results.items():
        print(f"{test}: {result['status']} - {result['message']}")
    json.dump(test_results, open(log_output_file, "w"), indent=2)